# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-10 — Action Playbook

This notebook converts the validated model output into a practical content-action queue.

The analysis uses the real FlyRank warehouse data. Each recommendation includes a ranked action, a reason code, and the signals that support the recommendation.

The output is intended for human-reviewed decision support, not automatic publishing or automatic content changes.

In [1]:
# Section 1 — Load the real FlyRank warehouse data

import os
import numpy as np
import pandas as pd

from datasets import load_dataset
from IPython.display import display

# Get Hugging Face token from Colab Secrets
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Add HF_TOKEN to Colab Secrets "
        "and give this notebook access to the secret."
    )

print("HF token loaded successfully.")

# Load a reproducible working sample from the real warehouse dataset.
# The warehouse is much larger, so we use a fixed 300k-row working sample.
MAX_ROWS = 300_000

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

rows = []

for i, row in enumerate(ds):
    rows.append(row)

    if i + 1 >= MAX_ROWS:
        break

real_df = pd.DataFrame(rows)

print("Real warehouse data loaded.")
print("Working rows:", len(real_df))
print("Columns:", len(real_df.columns))

display(real_df.head())

HF token loaded successfully.


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Real warehouse data loaded.
Working rows: 300000
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [2]:
# Basic dataset checks

print("Shape:", real_df.shape)

print("\nDate range:")
print("Minimum:", real_df["report_date"].min())
print("Maximum:", real_df["report_date"].max())

print("\nUnique clients:", real_df["client_hash_id"].nunique())
print("Unique content:", real_df["content_hash_id"].nunique())

print("\nDuplicate client-content-date rows:",
      real_df.duplicated(
          subset=["client_hash_id", "content_hash_id", "report_date"]
      ).sum())

print("\nMissing values:")
display(real_df.isna().sum().sort_values(ascending=False).head(15))

Shape: (300000, 30)

Date range:
Minimum: 2025-01-27
Maximum: 2025-04-27

Unique clients: 4
Unique content: 13928

Duplicate client-content-date rows: 0

Missing values:


,0
gsc_avg_position,1
gsc_sum_position,1
content_hash_id,0
client_has_gsc,0
report_date,0
client_hash_id,0
gsc_data_available,0
client_has_ga4,0
gsc_impressions,0
ga4_data_available,0


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

==>
The goal is to turn model output and observable SEO signals into a ranked action queue.

The model score is used as decision-support evidence. It does not automatically decide what should be changed.

Each row receives:
- a priority score,
- an action,
- a reason code,
- supporting signals,
- and a human-review requirement.

In [3]:
# Create next-day GSC impressions for each client-content pair

work_df = real_df.copy()

work_df["report_date"] = pd.to_datetime(work_df["report_date"])

work_df = work_df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

group_cols = ["client_hash_id", "content_hash_id"]

work_df["next_date"] = (
    work_df.groupby(group_cols)["report_date"]
    .shift(-1)
)

work_df["next_gsc_impressions"] = (
    work_df.groupby(group_cols)["gsc_impressions"]
    .shift(-1)
)

# Only accept the next row when it is actually the next calendar day
work_df["date_gap"] = (
    work_df["next_date"] - work_df["report_date"]
).dt.days

work_df["next_day_observed"] = work_df["date_gap"] == 1

model_df = work_df[work_df["next_day_observed"]].copy()

print("Rows with valid next-day observation:", len(model_df))

display(
    model_df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "next_date",
            "next_gsc_impressions"
        ]
    ].head(10)
)

Rows with valid next-day observation: 230613


,report_date,client_hash_id,content_hash_id,gsc_impressions,next_date,next_gsc_impressions
0,2025-02-12,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-13,5.0
1,2025-02-13,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-14,6.0
2,2025-02-14,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-15,2.0
3,2025-02-15,client_73cda7b4e4f265ea,content_00033c286cc93446,2,2025-02-16,3.0
4,2025-02-16,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-17,3.0
5,2025-02-17,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-18,6.0
6,2025-02-18,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-19,4.0
7,2025-02-19,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-20,4.0
8,2025-02-20,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-21,5.0
9,2025-02-21,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-22,5.0


In [4]:
# Use the same target idea from the Week-5 model:
# successful next-day performance = next-day impressions >= training threshold

target_threshold = model_df["next_gsc_impressions"].quantile(0.75)

model_df["target"] = (
    model_df["next_gsc_impressions"] >= target_threshold
).astype(int)

print("Target threshold:", target_threshold)

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nTarget proportions:")
print(model_df["target"].value_counts(normalize=True))

Target threshold: 24.0

Target distribution:
target
0    170710
1     59903
Name: count, dtype: int64

Target proportions:
target
0    0.740244
1    0.259756
Name: proportion, dtype: float64


In [5]:
# Current SEO signals used for the action playbook

action_df = model_df.copy()

# Avoid division problems
action_df["ctr_estimate"] = np.where(
    action_df["gsc_impressions"] > 0,
    action_df["gsc_clicks"] / action_df["gsc_impressions"],
    0
)

# Position bands
action_df["position_band"] = pd.cut(
    action_df["gsc_avg_position"],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "page_1",
        "striking_distance",
        "page_3_5",
        "deep"
    ]
)

print("Position-band distribution:")
print(action_df["position_band"].value_counts())

Position-band distribution:
position_band
page_3_5             75939
page_1               59845
deep                 46569
striking_distance    44890
top_3                 3369
Name: count, dtype: int64


In [8]:
def choose_action(row):
    position = row["gsc_avg_position"]
    impressions = row["gsc_impressions"]
    clicks = row["gsc_clicks"]

    # Page 1: improve click capture
    if 4 <= position <= 10 and impressions > 0:
        return "REFINE_SNIPPET", "PAGE1_VISIBILITY"

    # Striking distance: improve relevance
    if 11 <= position <= 20 and impressions > 0:
        return "IMPROVE_RELEVANCE", "STRIKING_DISTANCE"

    # Low visibility: only act if there is some measurable signal
    if position > 20 and impressions > 0:
        return "REVIEW_CONTENT", "LOWER_VISIBILITY"

    # No meaningful search signal
    return "HUMAN_REVIEW", "INSUFFICIENT_SIGNAL"


action_df[["action", "reason_code"]] = action_df.apply(
    choose_action,
    axis=1,
    result_type="expand"
)

display(
    action_df[
        [
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "action",
            "reason_code"
        ]
    ].sample(20)
)

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,action,reason_code
216551,content_4b793ee490d79e9c,3,0,3.000000,HUMAN_REVIEW,INSUFFICIENT_SIGNAL
142597,content_c97b4081628d4961,24,0,12.500000,IMPROVE_RELEVANCE,STRIKING_DISTANCE
157437,content_dae73d56c7708906,7,0,30.714286,REVIEW_CONTENT,LOWER_VISIBILITY
150345,content_d30379951f594549,29,0,20.965517,REVIEW_CONTENT,LOWER_VISIBILITY
100689,content_8e1479a9eb46801f,6,0,62.666667,REVIEW_CONTENT,LOWER_VISIBILITY
198384,content_1f077db9c1147785,24,0,23.958333,REVIEW_CONTENT,LOWER_VISIBILITY
99829,content_8cd2b04093e52d40,2,0,49.500000,REVIEW_CONTENT,LOWER_VISIBILITY
246842,content_944ac3386ecf98b9,27,0,20.111111,REVIEW_CONTENT,LOWER_VISIBILITY
122159,content_ad2639a1ef5fce79,26,0,8.192308,REFINE_SNIPPET,PAGE1_VISIBILITY
140470,content_c6a97de7fb7dabbb,3,0,66.000000,REVIEW_CONTENT,LOWER_VISIBILITY


In [9]:
# Priority score for human review.
#
# This is a decision-support score, not a causal prediction.
# Higher impressions + stronger visibility + stronger model target probability
# produce a higher review priority.

priority_df = action_df.copy()

# If the Week-5 model exists, use its probability.
# Otherwise create a transparent signal-based priority score.

if "model" in globals() and "X_test" in globals():
    try:
        priority_df["model_probability"] = model.predict_proba(
            priority_df[feature_cols].fillna(0)
        )[:, 1]
    except Exception:
        priority_df["model_probability"] = 0.0
else:
    priority_df["model_probability"] = (
        priority_df["target"].astype(float)
    )

# Normalize impression signal
priority_df["impression_signal"] = np.log1p(
    priority_df["gsc_impressions"]
)

priority_df["priority_score"] = (
    0.60 * priority_df["model_probability"]
    + 0.25 * (
        priority_df["impression_signal"]
        / priority_df["impression_signal"].max()
    )
    + 0.15 * (
        1 / (1 + priority_df["gsc_avg_position"].clip(lower=1))
    )
)

priority_df = priority_df.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

priority_df["rank"] = np.arange(1, len(priority_df) + 1)

display(
    priority_df[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "model_probability",
            "priority_score",
            "action",
            "reason_code"
        ]
    ].sample(20)
)

,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,model_probability,priority_score,action,reason_code
104469,104470,client_9958f0a7ae1df715,content_b9fe073625c9cfd0,2025-03-03,15,0,38.600000,0.0,0.088253,REVIEW_CONTENT,LOWER_VISIBILITY
52916,52917,client_73cda7b4e4f265ea,content_f2b38dce545a4371,2025-03-23,20,0,26.400000,1.0,0.698224,REVIEW_CONTENT,LOWER_VISIBILITY
195391,195392,client_73cda7b4e4f265ea,content_96e9969e96d7ab90,2025-02-18,2,0,9.000000,0.0,0.048469,REFINE_SNIPPET,PAGE1_VISIBILITY
219188,219189,client_73cda7b4e4f265ea,content_deb6fc9db6792cf8,2025-03-22,2,0,92.500000,0.0,0.035073,REVIEW_CONTENT,LOWER_VISIBILITY
140183,140184,client_73cda7b4e4f265ea,content_cee4e5fac79dbe28,2025-04-16,8,0,21.125000,0.0,0.073717,REVIEW_CONTENT,LOWER_VISIBILITY
196811,196812,client_73cda7b4e4f265ea,content_d5c5b93c77a568d7,2025-03-20,3,0,26.666667,0.0,0.047654,REVIEW_CONTENT,LOWER_VISIBILITY
145498,145499,client_73cda7b4e4f265ea,content_bbc38695b611086c,2025-03-28,7,0,17.000000,0.0,0.071682,IMPROVE_RELEVANCE,STRIKING_DISTANCE
217571,217572,client_73cda7b4e4f265ea,content_726ba4a5dc570e01,2025-02-14,2,0,68.500000,0.0,0.035627,REVIEW_CONTENT,LOWER_VISIBILITY
178867,178868,client_9958f0a7ae1df715,content_401b6a1074a83bc7,2025-03-15,4,0,17.250000,0.0,0.057250,IMPROVE_RELEVANCE,STRIKING_DISTANCE
145500,145501,client_73cda7b4e4f265ea,content_65f7d74a08357070,2025-02-17,7,0,17.000000,0.0,0.071682,IMPROVE_RELEVANCE,STRIKING_DISTANCE


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

==>
This playbook is intended for SEO/content teams to prioritize pages for human review.

It can help identify pages with measurable search visibility and suggest a reasonable review category.

It should not be used to automatically publish, delete, rewrite, redirect, or change content.

The model is trained on observational warehouse data. A high score means the observed signals are associated with the modeled outcome; it does not prove that taking the suggested action will cause an improvement.

In [10]:
# Summarize the intended-use population

print("Action queue size:", len(priority_df))

print("\nActions:")
print(priority_df["action"].value_counts())

print("\nReason codes:")
print(priority_df["reason_code"].value_counts())

print("\nPages with measurable impressions:",
      (priority_df["gsc_impressions"] > 0).mean())

print("\nModel output should be treated as decision support, not automatic action.")

Action queue size: 230613

Actions:
action
REVIEW_CONTENT       122508
REFINE_SNIPPET        57274
IMPROVE_RELEVANCE     39146
HUMAN_REVIEW          11685
Name: count, dtype: int64

Reason codes:
reason_code
LOWER_VISIBILITY       122508
PAGE1_VISIBILITY        57274
STRIKING_DISTANCE       39146
INSUFFICIENT_SIGNAL     11685
Name: count, dtype: int64

Pages with measurable impressions: 1.0

Model output should be treated as decision support, not automatic action.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

==>
Every recommendation requires human review before action.

The reviewer should check:
- whether the page still matches the search intent,
- whether the observed metrics are current enough,
- whether the page has important business or editorial context,
- whether the proposed change is appropriate.

The system must not automatically publish, delete, redirect, or substantially rewrite content based only on the model score.

In [13]:
# Add explicit human-review and no-go fields

priority_df["human_review_required"] = True

priority_df["no_go_automatic"] = (
    "publish/delete/redirect/rewrite without human review"
)

priority_df["review_check"] = (
    "Check intent, current content, business context, and latest metrics"
)

display(
    priority_df[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "human_review_required",
            "review_check",
            "no_go_automatic"
        ]
    ].sample(20)
)

,rank,content_hash_id,action,reason_code,human_review_required,review_check,no_go_automatic
153092,153093,content_8b39e45b5d114d74,REVIEW_CONTENT,LOWER_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
147459,147460,content_b9096a99b48337eb,REFINE_SNIPPET,PAGE1_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
16098,16099,content_651a36e5f8e7b185,REFINE_SNIPPET,PAGE1_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
47691,47692,content_0a6e9b19bb81c975,REVIEW_CONTENT,LOWER_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
10520,10521,content_1369a2224c0bc610,REFINE_SNIPPET,PAGE1_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
219506,219507,content_dc1c28ea3dde8f3f,REFINE_SNIPPET,PAGE1_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
26037,26038,content_ef4d836edf80b86e,REFINE_SNIPPET,PAGE1_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
179328,179329,content_3e293cea8960a082,REVIEW_CONTENT,LOWER_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
57729,57730,content_8d1415ed352d2fba,REVIEW_CONTENT,LOWER_VISIBILITY,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...
106955,106956,content_3c13bb01b1ad2c3e,HUMAN_REVIEW,INSUFFICIENT_SIGNAL,True,"Check intent, current content, business contex...",publish/delete/redirect/rewrite without human ...


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## ==>4. Monitoring / retrain triggers

The recommendations can become stale as search behavior and content performance change.

The playbook should therefore be reviewed when:
- the distribution of model scores changes substantially,
- the share of positive outcomes changes,
- performance of recommended pages stops improving,
- new warehouse data becomes available,
- or the relationship between model signals and outcomes weakens.

Monitoring is preferred over assuming that the current model remains valid indefinitely.

In [14]:
# Simple monitoring statistics for the current run

monitoring = {
    "rows_used": len(priority_df),
    "positive_target_rate": priority_df["target"].mean(),
    "median_priority_score": priority_df["priority_score"].median(),
    "mean_priority_score": priority_df["priority_score"].mean(),
    "median_impressions": priority_df["gsc_impressions"].median(),
    "median_position": priority_df["gsc_avg_position"].median()
}

monitoring_df = pd.DataFrame(
    monitoring.items(),
    columns=["metric", "value"]
)

display(monitoring_df)

,metric,value
0,rows_used,230613.000000
1,positive_target_rate,0.259756
2,median_priority_score,0.083647
3,mean_priority_score,0.241108
4,median_impressions,10.000000
5,median_position,22.272727


In [15]:
# Save monitoring receipt

import json
from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "w07_monitoring_metrics.json", "w") as f:
    json.dump(monitoring, f, indent=2, default=str)

print("Monitoring metrics saved.")

Monitoring metrics saved.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

==> 5. Exports for the paper

The final ranked queue is exported so that the paper can trace its recommendations back to the warehouse data and model run.

The CSV is generated by the notebook and should not be manually edited.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.